# Laboratorio Final — Starbucks Rewards: del experimento a la decisión

Cierra el bloque de Churn y Uplift Modeling. Van a trabajar con la campaña COMPLETA (no una
muestra de prueba): 40,000 clientes, mitad recibió un cupón de $5 (tratamiento), mitad no
(control), medidos a 60 días.

Este laboratorio repite, sobre la base completa, lo que ya practicaron en los Breakout Rooms:

| Dónde lo vieron | Qué van a repetir aquí |
|---|---|
| Breakout 2 — Las dos ventanas | Un modelo de riesgo/churn, para targetear y ver la pérdida |
| Breakout 3 — Los 4 cuadrantes | El T-learner: dos modelos, uno por grupo |
| Breakout 4 — La curva Qini | Miden qué tan bien ordena su T-learner con datos que nunca vio |

Las columnas `recency_days`, `frequency` y `monetary` son las mismas del RFM de la Semana 1.

**Los 3 pasos del laboratorio:**
1. Entrenan un modelo de churn y lo usan para targetear — van a ver la pérdida con sus
   propios números.
2. Programan el T-learner y miden su Qini.
3. Deciden a quién contactar: eligen el porcentaje óptimo y defienden la cifra.

El dataset trae una columna oculta con el **tipo real de cada cliente y su uplift
verdadero** — en la vida real esa columna no existe. Al final del laboratorio la van a
destapar para ver si sus modelos acertaron.

Donde dice **`# TODO`** hay algo para que ustedes completen.


**Grupo 1.** Fuente: material oficial de Semana_3/Tarea. Datos educativos simulados. La verdad oculta se consulta sólo en la revelación final. La partición y funciones originales se conservan; el escalado se aprende dentro de cada brazo con entrenamiento exclusivamente.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display
BASE = Path.cwd() if (Path.cwd()/'analisis.py').exists() else Path.cwd()/'Semana_3'
sys.path.insert(0,str(BASE))
from analisis import ejecutar, revelar, figura, politica, FEATURES, SEED
r = ejecutar(revelar_verdad=False)
train, test = r['train'], r['test']
print(f'{len(train)+len(test):,} clientes; entrenamiento {len(train):,}; evaluación {len(test):,}; semilla {SEED}')
display(test.head())

40,000 clientes; entrenamiento 28,000; evaluación 12,000; semilla 42


,customer_id,treatment_group,recency_days,frequency,monetary,email_open_rate,tenure_days,responded_60d,value_60d,margin_60d,utilidad_neta_60d,prob_churn,uplift_Logística,uplift_Árbol sin límite,uplift_Árbol max_depth=2,uplift_Random Forest,uplift_estimado
414,415,treatment,24,11,473.92,0.287,763,0,0.00,0.00,-0.05,0.335401,-0.045473,1.0,-0.019990,0.098615,-0.045473
37024,37025,treatment,8,22,405.91,0.567,903,0,0.00,0.00,-0.05,0.203782,0.002115,0.0,-0.019990,-0.036828,0.002115
29140,29141,control,18,17,358.39,0.318,50,1,24.44,6.11,6.11,0.316233,-0.004737,0.0,-0.019990,0.038687,-0.004737
28890,28891,control,69,6,181.34,0.450,603,1,58.42,14.61,14.61,0.655117,0.188717,1.0,0.332084,0.322492,0.188717
30111,30112,treatment,248,2,26.21,0.309,807,0,0.00,0.00,-0.05,0.954925,0.052860,0.0,-0.065442,-0.023757,0.052860


## Paso 1 — entrenen un modelo de churn y usen para targetear

Un modelo de churn normal: no sabe nada de tratamiento/control, solo aprende a predecir quién
NO va a volver a comprar. Lo van a usar para elegir a quién contactar (targeting por riesgo) y
van a medir qué tan bien o mal les fue con esa decisión, usando el experimento real como juez.


In [2]:
print('Predictores anteriores al tratamiento:', FEATURES)
print('AUC churn:', r['auc_riesgo'])
print('Política de riesgo al 30%:',r['riesgo30'])
assert set(train.customer_id).isdisjoint(test.customer_id)

Predictores anteriores al tratamiento: ['recency_days', 'frequency', 'monetary', 'email_open_rate', 'tenure_days']
AUC churn: 0.8161067494827721
Política de riesgo al 30%: {'pct_contactado': 30, 'valor_por_cliente': np.float64(0.06468499128503924), 'n_contactados': 3600, 'valor_total': np.float64(232.86596862614127)}


**Respuesta Paso 1.** El churn se define como 1 − responded_60d. La logística aprende en entrenamiento, sin variables de resultados, tratamiento o verdad como predictores. Su AUC mide capacidad predictiva, no causalidad. El valor de la política se estima comparando utilidad neta de tratados y margen de controles del mismo segmento; el resultado numérico está arriba. El riesgo combina respuestas de ambos brazos según el enunciado, por lo que no equivale al riesgo sin campaña.

## Paso 2 — programen el T-learner y midan su Qini

Dos modelos de **Regresión Logística**: uno entrenado SOLO con los clientes tratados de entrenamiento, otro SOLO con los
de control. Le preguntan a los dos sobre el conjunto de prueba completo, y restan.


In [3]:
import inspect
import analisis
print(inspect.getsource(analisis.modelos))
print(inspect.getsource(analisis.ejecutar))
print('Qini logística:',r['comparacion'].set_index('modelo').loc['Logística','indice_qini'])
print('Qini riesgo:',r['qini_riesgo'])

def modelos():
    return {
        'Logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
        'Árbol sin límite': DecisionTreeClassifier(random_state=SEED),
        'Árbol max_depth=2': DecisionTreeClassifier(max_depth=2, random_state=SEED),
        'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=50, random_state=SEED, n_jobs=2),
    }

def ejecutar(revelar_verdad=True):
    df = pd.read_csv(BASE/'data/uplift_campaign.csv')
    assert len(df) == 40000 and df.customer_id.is_unique
    assert not df[FEATURES].isna().any().any()
    train, test = train_test_split(df, test_size=.30, random_state=SEED, stratify=df.treatment_group)
    assert set(train.customer_id).isdisjoint(test.customer_id)
    test = test.copy()
    riesgo = clone(modelos()['Logística']).fit(train[FEATURES], 1-train.responded_60d)
    test['prob_churn'] = riesgo.predict_proba(test[FEATURES])[:,1]
    filas, curvas = [], {}
    for n

**Respuesta Paso 2.** Cada T-learner estima p(Y=1|X,T=1) y p(Y=1|X,T=0) en brazos separados y resta ambas probabilidades sobre la misma evaluación. El StandardScaler de cada logística se ajusta exclusivamente en su brazo de entrenamiento. Los cuatro clasificadores y sus hiperparámetros se fijaron antes de consultar la verdad. No se utilizan identificadores, margen, utilidad ni respuesta como atributos.

In [4]:
print(inspect.getsource(analisis.curva_qini))
print(inspect.getsource(analisis.coeficiente_qini))
fig=figura(r)
display(fig)

def curva_qini(test_df, score_col, steps=40):
    # Conserva exactamente la definición y muestreo del material original.
    orden = test_df.sort_values(score_col, ascending=False).reset_index(drop=True)
    tratado = (orden.treatment_group == 'treatment').values
    y = orden.responded_60d.values
    rt, rc = np.cumsum(np.where(tratado, y, 0)), np.cumsum(np.where(~tratado, y, 0))
    nt, nc = np.cumsum(tratado), np.cumsum(~tratado)
    n = len(orden)
    xs, ys = [0.0], [0.0]
    for i in np.linspace(1, n, steps).astype(int):
        t = rt[i-1] / nt[i-1] if nt[i-1] else 0
        c = rc[i-1] / nc[i-1] if nc[i-1] else 0
        xs.append(i/n)
        ys.append((t-c)*n)
    return np.array(xs), np.array(ys)

def coeficiente_qini(xs, ys, n):
    return (np.trapezoid(ys, xs) - np.trapezoid(np.linspace(0, ys[-1], len(xs)), xs))/n



<Figure size 900x500 with 1 Axes>

**Definición de Qini conservada.** En cada prefijo se calcula la tasa de respuesta tratada menos la tasa control y se multiplica por N, el tamaño completo de evaluación. Se muestrean 40 posiciones más el origen. El índice es [área trapezoidal de la curva − área trapezoidal de np.linspace(0, y_final, len(xs)) sobre los mismos xs] / N. Se mantiene incluso la referencia discretizada del notebook. Esta curva no es el conteo acumulado estándar: usa N, no el tamaño del prefijo. El primer prefijo puede contener un solo brazo y crea inestabilidad; con árboles también hay empates. Un Qini mayor señala mejor ranking bajo esta convención, sin implicar mayor utilidad monetaria.

## Paso 3 — decidan a quién contactar

Prueben varios porcentajes de la base (10%, 20%, ..., 100%) ordenados por `uplift_estimado`, y
para cada uno calculen el valor neto total (usando `value_60d`, igual que en el Paso 2).
Encuentren el porcentaje que deja más dinero sobre la mesa.


In [5]:
print(inspect.getsource(analisis.valor_de_la_politica))
display(r['tabla'].round(2))
mejor=r['tabla'].loc[r['tabla'].valor_total.idxmax()]
unitario=r['tabla'].loc[r['tabla'].valor_por_cliente.idxmax()]
print('Óptimo total:',mejor.to_dict())
print('Óptimo unitario:',unitario.to_dict())

def valor_de_la_politica(test_df, seleccionados):
    sel = test_df.loc[seleccionados]
    vt = sel.loc[sel.treatment_group == 'treatment', 'utilidad_neta_60d'].mean()
    vc = sel.loc[sel.treatment_group == 'control', 'margin_60d'].mean()
    return vt-vc, len(sel)



,pct_contactado,valor_por_cliente,n_contactados,valor_total
0,10,1.61,1200,1929.60
1,20,0.95,2400,2275.78
2,30,0.47,3600,1675.50
3,40,-0.12,4800,-595.14
4,50,-0.50,6000,-2976.99
5,60,-0.73,7200,-5255.47
6,70,-0.94,8400,-7873.05
7,80,-1.17,9600,-11217.50
8,90,-1.45,10800,-15681.29
9,100,-1.69,12000,-20236.13


Óptimo total: {'pct_contactado': 20.0, 'valor_por_cliente': 0.9482403813623197, 'n_contactados': 2400.0, 'valor_total': 2275.7769152695673}
Óptimo unitario: {'pct_contactado': 10.0, 'valor_por_cliente': 1.6079970815092768, 'n_contactados': 1200.0, 'valor_total': 1929.5964978111322}


**Pregunta para defender con su propia cifra:** ¿cuál % maximiza el valor TOTAL? ¿Es el mismo
que maximiza el valor POR CLIENTE? ¿Por qué no, necesariamente?


**Respuesta.** El máximo total corresponde al 20%, con $+2,275.78; el máximo por contactado corresponde al 10%. No necesariamente coinciden: el valor total multiplica el valor unitario por el volumen. Se mantienen los valores sin redondear para elegir el máximo. Aunque el texto menciona value_60d, la función original y el starter utilizan utilidad_neta_60d frente a margin_60d: seguimos esa definición económica. La selección entre diez porcentajes es exploratoria y su máximo necesita evaluación independiente. Los totales no representan una ganancia ya realizada ni toda la base de 40.000.

## Paso 4 — Auditoría del diseño

La muestra de evaluación es idéntica para los cuatro modelos (12.000 clientes); entrenamiento tiene 28.000. No hay fuga hacia el ajuste de modelos: se utiliza una lista cerrada de cinco atributos previos y escalado dentro de cada brazo. La búsqueda de política y comparación sobre evaluación sí generan sesgo de selección; no se reutiliza esta muestra como evidencia confirmatoria de la política seleccionada. Se exige una nueva prueba. Los datos son simulados con verdad conocida, no resultados comerciales verificables.

In [6]:
print(train.treatment_group.value_counts())
print(test.treatment_group.value_counts())
assert not any(c.startswith('true_') for c in FEATURES)
assert test.customer_id.is_unique
assert np.isfinite(r['comparacion'][['indice_qini']]).all().all()

treatment_group
control      14055
treatment    13945
Name: count, dtype: int64
treatment_group
control      6024
treatment    5976
Name: count, dtype: int64


## Paso 5 — La revelación final y comparación de modelos

## La revelación final

Esta es la parte que no pudieron hacer en los Breakout Rooms: el dataset completo
(`uplift_campaign_FULL.csv`) trae el **tipo real** de cada cliente y su **uplift verdadero** —
información que en la vida real nunca existe. Compárenla contra lo que su modelo estimó.


La verdad se une por customer_id con validación uno a uno. La correlación es Pearson contra true_uplift_prob, nunca contra la respuesta observada. Se conserva la logística como política principal del Paso 3; los otros modelos son una comparación exploratoria.

In [7]:
revelar(r)
verdad=r['verdad']
display((verdad.true_type.value_counts(normalize=True)*100).round(2))
display(r['comparacion'])
r['comparacion'].to_csv(BASE/'comparacion_modelos.csv',index=False)
seleccion=verdad.sort_values('uplift_estimado',ascending=False).head(int(mejor.n_contactados))
oracle=verdad.sort_values('true_value_of_contact',ascending=False).head(int(mejor.n_contactados))
print('Valor esperado verdadero de la política:',seleccion.true_value_of_contact.sum())
print('Valor esperado con conocimiento perfecto, mismo volumen:',oracle.true_value_of_contact.sum())
print('Composición de contactos elegidos:')
display((seleccion.true_type.value_counts(normalize=True)*100).round(2))

true_type
sure_thing      48.88
persuadable     22.43
lost_cause      21.52
sleeping_dog     7.16
Name: proportion, dtype: float64

,modelo,correlacion_uplift_real,indice_qini
0,Logística,0.618635,0.141527
1,Árbol sin límite,0.203512,0.065821
2,Árbol max_depth=2,0.605265,0.126014
3,Random Forest,0.862467,0.157588


Valor esperado verdadero de la política: 2835.6299999999997
Valor esperado con conocimiento perfecto, mismo volumen: 5423.999999999998
Composición de contactos elegidos:


true_type
persuadable     79.75
sure_thing      12.71
lost_cause       4.46
sleeping_dog     3.08
Name: proportion, dtype: float64

**Respuesta comparación.** La correlación logística es 0.6186. El mayor Qini corresponde a Random Forest, con +0.1576. La tabla muestra qué modelo mejora cada criterio; correlación y Qini no evalúan exactamente lo mismo y no implican calibración perfecta ni beneficio económico. El árbol sin límite puede memorizar entrenamiento y producir diferencias extremas de probabilidades; profundidad y tamaño mínimo de hoja regularizan el Random Forest. La auditoría de valor compara efectos esperados simulados, mientras el Paso 3 compara resultados observados aleatorios: son estimandos relacionados pero no números idénticos.

# Conclusión dirigida al Presidente de Marketing

Señor Presidente de Marketing: recomendamos evaluar en un nuevo experimento una política de contacto dirigida al 20% de la base, ordenada por el uplift del T-learner logístico. Esta recomendación proviene de la campaña educativa de Starbucks Rewards: 40.000 clientes, cupón de $5 y medición a 60 días. Los datos son simulados, con tipos y efectos verdaderos disponibles para auditoría; no representan evidencia comercial real de Starbucks.

En los 12.000 clientes reservados para evaluación, esa política contacta a 2,400 personas y produce una estimación incremental neta de $+0.95 por contactado, equivalente a $+2,275.78 en esta muestra. Es el máximo de los diez porcentajes examinados. El mayor valor unitario aparece al 10%: priorizar rentabilidad por persona y maximizar valor total son objetivos distintos. Contactar al 30% de mayor riesgo de churn estima $+232.87; predecir quién no compra no identifica necesariamente a quien cambia su comportamiento por el cupón.

El T-learner logístico obtiene Qini +0.1415, frente a +0.0476 del ranking por riesgo. Tras revelar la verdad, la correlación de Pearson entre uplift estimado y verdadero es 0.6186. En la comparación de cuatro clasificadores, Random Forest alcanza el mayor Qini (+0.1576); la tabla adjunta permite contrastar también la correlación. Qini evalúa ordenamiento de respuestas, no ganancia económica, y su fórmula aquí es la específica del laboratorio.

La auditoría final aporta una referencia adicional: los contactos elegidos por logística suman $+2,835.63 de valor esperado verdadero, frente a $+5,424.00 si pudiéramos elegir con conocimiento perfecto, para igual número de contactos. Esta diferencia cuantifica lo que deja de capturar el modelo. La verdad se usó sólo después de entrenar y evaluar; nunca como predictor ni para elegir el porcentaje.

La recomendación supone asignación aleatoria, comparabilidad de tratados y controles, ausencia de interferencia entre clientes y atributos medidos antes de la campaña. La utilidad descuenta $5 sólo cuando hay compra y $0,05 por envío. No incluye efectos posteriores a 60 días, capacidad operativa ni costos adicionales. Se entrenó en el 70% de la base con semilla fija; la búsqueda del porcentaje sobre el mismo 30% de evaluación puede sobreestimar su beneficio, y la comparación de modelos también es exploratoria. No presentamos estos máximos como una prueba independiente de desempeño.

Antes de extender la política, recomendamos una prueba aleatoria nueva en el porcentaje propuesto, con un control sin cupón y un criterio de utilidad neta acordado previamente. La extrapolación a toda la base exige estabilidad de clientes, costos y efectos; los importes reportados corresponden únicamente a la evaluación. El avance ejecutivo consiste en priorizar valor incremental, validar la decisión y evitar gastar en clientes que comprarían de todos modos.

## Reproducibilidad y entregables

Ejecutar desde la raíz o Semana_3 con las dependencias de Semana_3/requirements.txt. analisis.py comparte exactamente las funciones con Streamlit. La entrega incluye comparacion_modelos.csv y conclusion.pdf; la tabla de políticas y la curva Qini quedan dentro del notebook y de la aplicación. Fuente original: https://github.com/stanlybolanos/marketing-analytics-base-2026/tree/main/Semana_3/Tarea. No se sustituyeron los datos ni se copiaron las métricas de ejemplo del enunciado.